In [1]:

import pandas as pd
import requests

print("Python funktioniert!")
print("Pandas-Version:", pd.__version__)
print("Requests-Version:", requests.__version__)


Python funktioniert!
Pandas-Version: 3.1.1
Requests-Version: 2.34.2


In [3]:

# Open-Meteo Historical Weather API
url = "https://archive-api.open-meteo.com/v1/archive"

params = {
    "latitude": 47.3769,
    "longitude": 8.5417,
    "start_date": "2023-01-01",
    "end_date": "2023-01-07",
    "hourly": [
        "temperature_2m",
        "precipitation",
        "relative_humidity_2m",
        "wind_speed_10m",
        "cloud_cover"
    ],
    "models": "era5",
    "timezone": "UTC"
}

response = requests.get(
    url,
    params=params,
    timeout=60
)

response.raise_for_status()

data = response.json()

df_weather = pd.DataFrame(data["hourly"])

print("Anzahl Zeilen:", len(df_weather))

df_weather.head()


Anzahl Zeilen: 168


,time,temperature_2m,precipitation,relative_humidity_2m,wind_speed_10m,cloud_cover
0,2023-01-01T00:00,8.1,0.0,84,5.1,74
1,2023-01-01T01:00,8.2,0.0,83,4.6,48
2,2023-01-01T02:00,8.4,0.0,85,3.6,48
3,2023-01-01T03:00,8.0,0.0,86,2.9,61
4,2023-01-01T04:00,8.0,0.0,86,3.2,41


In [4]:

# Zeitraum: 2023 bis 2025
years = [2023, 2024, 2025]

# Liste für die Wetterdaten aller Jahre
weather_data = []

for year in years:

    print(f"Lade Wetterdaten für {year}...")

    # Parameter aus dem erfolgreichen API-Test übernehmen
    year_params = params.copy()

    year_params["start_date"] = f"{year}-01-01"
    year_params["end_date"] = f"{year}-12-31"

    # API aufrufen
    response = requests.get(
        url,
        params=year_params,
        timeout=120
    )

    response.raise_for_status()

    data = response.json()

    # Wetterdaten in DataFrame umwandeln
    df_year = pd.DataFrame(data["hourly"])

    # Jahr zur Kontrolle ausgeben
    print(f"{year}: {len(df_year)} Zeilen")

    weather_data.append(df_year)

# Alle drei Jahre zusammenführen
df_weather = pd.concat(
    weather_data,
    ignore_index=True
)

print("\nGesamtzahl Zeilen:", len(df_weather))

# Erste fünf Zeilen anzeigen
df_weather.head()


Lade Wetterdaten für 2023...
2023: 8760 Zeilen
Lade Wetterdaten für 2024...
2024: 8784 Zeilen
Lade Wetterdaten für 2025...
2025: 8760 Zeilen

Gesamtzahl Zeilen: 26304


,time,temperature_2m,precipitation,relative_humidity_2m,wind_speed_10m,cloud_cover
0,2023-01-01T00:00,8.1,0.0,84,5.1,74
1,2023-01-01T01:00,8.2,0.0,83,4.6,48
2,2023-01-01T02:00,8.4,0.0,85,3.6,48
3,2023-01-01T03:00,8.0,0.0,86,2.9,61
4,2023-01-01T04:00,8.0,0.0,86,3.2,41


In [5]:

print("Anzahl Zeilen und Spalten:")
print(df_weather.shape)

print("\nSpaltennamen:")
print(df_weather.columns.tolist())

print("\nFehlende Werte:")
print(df_weather.isnull().sum())

print("\nDoppelte Zeitstempel:")
print(df_weather["time"].duplicated().sum())

print("\nErster Zeitstempel:")
print(df_weather["time"].min())

print("\nLetzter Zeitstempel:")
print(df_weather["time"].max())


Anzahl Zeilen und Spalten:
(26304, 6)

Spaltennamen:
['time', 'temperature_2m', 'precipitation', 'relative_humidity_2m', 'wind_speed_10m', 'cloud_cover']

Fehlende Werte:
time                    0
temperature_2m          0
precipitation           0
relative_humidity_2m    0
wind_speed_10m          0
cloud_cover             0
dtype: int64

Doppelte Zeitstempel:
0

Erster Zeitstempel:
2023-01-01T00:00

Letzter Zeitstempel:
2025-12-31T23:00


In [6]:

from pathlib import Path

# Projekt-Hauptordner finden
current = Path.cwd().resolve()

project_root = next(
    (
        p for p in [current, *current.parents]
        if (p / "requirements.txt").exists()
        and (p / "notebooks").is_dir()
    ),
    None
)

if project_root is None:
    raise FileNotFoundError(
        "Projekt-Hauptordner nicht gefunden!"
    )

# Ordner für Rohdaten
output_dir = project_root / "data" / "raw"
output_dir.mkdir(parents=True, exist_ok=True)

# Dateiname
file_path = (
    output_dir /
    "weather_zurich_hourly_2023_2025.csv"
)

# Wetterdaten speichern
df_weather.to_csv(
    file_path,
    index=False
)

print("Wetterdaten erfolgreich gespeichert!")
print("Speicherort:", file_path)
print("Anzahl Zeilen:", len(df_weather))


Wetterdaten erfolgreich gespeichert!
Speicherort: C:\Sven\Studium\Fächer\2. Semester Hauptstudium\Data Analytics\Projekt\velo-wetter-zuerich-HS26\velo-wetter-zuerich-HS26\data\raw\weather_zurich_hourly_2023_2025.csv
Anzahl Zeilen: 26304


In [7]:

# Gespeicherte Wetterdaten erneut einlesen
df_check = pd.read_csv(file_path)

print("Anzahl Zeilen:", len(df_check))
print("Anzahl Spalten:", len(df_check.columns))

# Erste fünf Zeilen anzeigen
df_check.head()


Anzahl Zeilen: 26304
Anzahl Spalten: 6


,time,temperature_2m,precipitation,relative_humidity_2m,wind_speed_10m,cloud_cover
0,2023-01-01T00:00,8.1,0.0,84,5.1,74
1,2023-01-01T01:00,8.2,0.0,83,4.6,48
2,2023-01-01T02:00,8.4,0.0,85,3.6,48
3,2023-01-01T03:00,8.0,0.0,86,2.9,61
4,2023-01-01T04:00,8.0,0.0,86,3.2,41
